# Notebook 3: Modellering & Dimensionsreduktion

---

Nu är datan förberedd. I det här steget **tränar vi flera klassificeringsmodeller**
och utvärderar dem med korsvalidering.

Vi testar:
1. Logistisk Regression
2. K-Närmaste Grannar (KNN)
3. Beslutsträd
4. Random Forest
5. PCA-reducerade features + LR och KNN
6. UMAP-reducerade features + LR och KNN

Resultaten sparas och jämförs i sin helhet i Notebook 4.

## Ladda data

Vi läser in den processade och skalade data som skapades i Notebook 2.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report
from sklearn.decomposition import PCA
from umap import UMAP

plt.rcParams['figure.figsize'] = (10, 6)
sns.set_theme(style='whitegrid', palette='muted')

# Ladda in processad data
X_train = pd.read_csv('Data/processed/X_train_scaled.csv').values
X_test  = pd.read_csv('Data/processed/X_test_scaled.csv').values
y_train = pd.read_csv('Data/processed/y_train.csv').values.ravel()
y_test  = pd.read_csv('Data/processed/y_test.csv').values.ravel()
feature_names = pd.read_csv('Data/processed/X_train_scaled.csv').columns.tolist()

print(f'Träningsdata: {X_train.shape}  |  Testdata: {X_test.shape}')

# Hjälpfunktion – beräknar och sparar metrics för en modell
resultat = {}  # Dict där vi samlar alla modellers resultat

def utvardera(namn, modell, X_tr, X_te, y_tr, y_te, cv_folds=5):
    """Tränar, utvärderar och sparar resultat för en modell."""
    modell.fit(X_tr, y_tr)
    y_pred = modell.predict(X_te)
    y_prob = modell.predict_proba(X_te)[:, 1]

    cv = StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=42)
    cv_f1 = cross_val_score(modell, X_tr, y_tr, cv=cv, scoring='f1').mean()

    resultat[namn] = {
        'Accuracy':  round(accuracy_score(y_te, y_pred), 4),
        'F1-score':  round(f1_score(y_te, y_pred), 4),
        'ROC-AUC':   round(roc_auc_score(y_te, y_prob), 4),
        'CV F1 (5-fold)': round(cv_f1, 4),
        'modell': modell,
        'y_pred': y_pred,
        'y_prob': y_prob
    }
    print(f'{namn:<35}  Accuracy={resultat[namn]["Accuracy"]:.4f}  '
          f'F1={resultat[namn]["F1-score"]:.4f}  AUC={resultat[namn]["ROC-AUC"]:.4f}  '
          f'CV-F1={resultat[namn]["CV F1 (5-fold)"]:.4f}')
    return modell

## Logistisk Regression

Logistisk Regression är en linjär modell som beräknar sannolikheten för att en patient
tillhör klassen `HeartDisease = 1`.

Den passar bra som **baslinje** – om mer komplexa modeller inte slår den,
är det ett tecken på att datan kanske är linjärt separerbar.

Koefficienter: positiv koefficient → variabeln ökar risken, negativ → minskar risken.

In [ ]:
lr = utvardera(
    'Logistisk Regression',
    LogisticRegression(max_iter=1000, random_state=42),
    X_train, X_test, y_train, y_test
)

# Visa koefficienter – vilka variabler påverkar mest?
koeff_df = pd.DataFrame({
    'Feature': feature_names,
    'Koefficient': lr.coef_[0]
}).sort_values('Koefficient', key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
farger = ['tomato' if v > 0 else 'steelblue' for v in koeff_df['Koefficient']]
ax.barh(koeff_df['Feature'], koeff_df['Koefficient'], color=farger, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Logistisk Regression – Koefficienter\n(röd = ökar risk, blå = minskar risk)',
             fontweight='bold')
ax.set_xlabel('Koefficient')
plt.tight_layout()
plt.savefig('outputs/figures/03_lr_koefficienter.png', dpi=150)
plt.show()

## KNN

KNN (K-Nearest Neighbors) klassificerar en patient baserat på de `k` närmaste grannarna
i träningsdatan. Majoritetens klass avgör.

`k` är en hyperparameter – vi testar flera värden och väljer det bästa via korsvalidering.

**Tumregel:** Litet `k` → känslig för brus (overfit), stort `k` → för generaliserande (underfit).

In [ ]:
# Sök bästa k med korsvalidering
k_variabler = range(1, 21)
cv_resultat_knn = []

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for k in k_variabler:
    knn_tmp = KNeighborsClassifier(n_neighbors=k)
    f1 = cross_val_score(knn_tmp, X_train, y_train, cv=cv, scoring='f1').mean()
    cv_resultat_knn.append(f1)

basta_k = k_variabler[np.argmax(cv_resultat_knn)]
print(f'Bästa k = {basta_k}  (CV F1 = {max(cv_resultat_knn):.4f})')

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(k_variabler, cv_resultat_knn, marker='o', color='steelblue')
ax.axvline(basta_k, color='tomato', linestyle='--', label=f'Bästa k={basta_k}')
ax.set_title('KNN – CV F1-score per k', fontweight='bold')
ax.set_xlabel('k (antal grannar)')
ax.set_ylabel('CV F1-score')
ax.legend()
plt.tight_layout()
plt.savefig('outputs/figures/03_knn_k_val.png', dpi=150)
plt.show()

In [ ]:
knn = utvardera(
    f'KNN (k={basta_k})',
    KNeighborsClassifier(n_neighbors=basta_k),
    X_train, X_test, y_train, y_test
)

## Beslutsträd

Ett beslutsträd skapar en serie ja/nej-frågor baserat på features, tills det hamnar
i ett löv som representerar en klass.

Fördelen: Beslutsträd är lätt att tolka – ni kan visualisera hela trädet.
Nackdelen: Det är lätt att **overfitta** (lära sig träningsdata för väl).

`max_depth` begränsar trädets djup för att undvika overfit. Vi söker bästa djup med CV.

In [ ]:
# Sök bästa max_depth
djup_variabler = range(1, 15)
cv_resultat_dt = []

for djup in djup_variabler:
    dt_tmp = DecisionTreeClassifier(max_depth=djup, random_state=42)
    f1 = cross_val_score(dt_tmp, X_train, y_train, cv=cv, scoring='f1').mean()
    cv_resultat_dt.append(f1)

basta_djup = djup_variabler[np.argmax(cv_resultat_dt)]
print(f'Bästa max_depth = {basta_djup}  (CV F1 = {max(cv_resultat_dt):.4f})')

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(djup_variabler, cv_resultat_dt, marker='o', color='steelblue')
ax.axvline(basta_djup, color='tomato', linestyle='--', label=f'Bästa djup={basta_djup}')
ax.set_title('Beslutsträd – CV F1-score per max_depth', fontweight='bold')
ax.set_xlabel('max_depth')
ax.set_ylabel('CV F1-score')
ax.legend()
plt.tight_layout()
plt.savefig('outputs/figures/03_dt_djup_val.png', dpi=150)
plt.show()

In [ ]:
dt = utvardera(
    f'Beslutsträd (depth={basta_djup})',
    DecisionTreeClassifier(max_depth=basta_djup, random_state=42),
    X_train, X_test, y_train, y_test
)

## Random Forest

Random Forest bygger ett **ensemble** av många beslutsträd, där varje träd tränas på
en slumpmässig delmängd av data och features. Den slutliga klassificeringen är
majoritetens röst bland alla träd.

Fördelen: Mycket robust mot overfit och hanterar brus bra.
Nackdelen: Svårare att tolka än ett enskilt beslutsträd.

**Feature Importance** berättar vilka variabler som bidrar mest till förutsägelserna.

In [ ]:
rf = utvardera(
    'Random Forest',
    RandomForestClassifier(n_estimators=200, random_state=42),
    X_train, X_test, y_train, y_test
)

# Feature importance – vilka variabler är viktigast?
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': rf.feature_importances_
}).sort_values('Importance', ascending=False).head(12)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(importance_df['Feature'][::-1], importance_df['Importance'][::-1],
        color='steelblue', edgecolor='white')
ax.set_title('Random Forest – Feature Importance (top 12)', fontweight='bold')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.savefig('outputs/figures/03_rf_feature_importance.png', dpi=150)
plt.show()

## Korsvalidering – Sammanfattning

Här är en samlad vy av CV F1-scorerna för alla fyra modeller hittills.
CV (5-fold) delar träningsdatan i 5 delar och tränar/testar 5 gånger – mer tillförlitligt
än att bara titta på ett enda test-set.

In [ ]:
# Sammanfattningstabell – alla modeller hittills
df_resultat = pd.DataFrame([
    {k: v for k, v in v_dict.items() if k not in ['modell', 'y_pred', 'y_prob']}
    for v_dict in resultat.values()
], index=resultat.keys())

print('=== Sammanfattning – alla modeller (hittills) ===')
df_resultat[['Accuracy', 'F1-score', 'ROC-AUC', 'CV F1 (5-fold)']].round(4)

---

## PCA – Principal Component Analysis

PCA är en teknik för **dimensionsreduktion**. Den skapar nya variabler (principal components)
som är linjära kombinationer av de ursprungliga och fångar den maximala variansen.

Varför använda PCA?
- Reducerar dimensioner → snabbare träning
- Tar bort multikollinearitet → kan hjälpa Logistisk Regression
- Kan förbättra eller försämra resultaten – det är empiriskt!

Vi väljer antal komponenter med ett **scree plot** – vid "knäet" avtar informationsvinsten.

In [ ]:
# Fit PCA på träningsdata – aldrig på testdata!
pca_full = PCA(random_state=42)
pca_full.fit(X_train)

# Kumulativ förklarad varians
kumulativ_varians = np.cumsum(pca_full.explained_variance_ratio_) * 100
n_komp_90 = np.argmax(kumulativ_varians >= 90) + 1

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scree plot
axes[0].bar(range(1, len(pca_full.explained_variance_ratio_) + 1),
           pca_full.explained_variance_ratio_ * 100, color='steelblue', edgecolor='white')
axes[0].set_title('PCA – Förklarad varians per komponent', fontweight='bold')
axes[0].set_xlabel('Komponent')
axes[0].set_ylabel('Förklarad varians (%)')

# Kumulativ varians
axes[1].plot(range(1, len(kumulativ_varians) + 1), kumulativ_varians,
             marker='o', color='steelblue')
axes[1].axhline(90, color='tomato', linestyle='--', label='90% tröskel')
axes[1].axvline(n_komp_90, color='tomato', linestyle=':', label=f'{n_komp_90} komponenter')
axes[1].set_title('PCA – Kumulativ förklarad varians', fontweight='bold')
axes[1].set_xlabel('Antal komponenter')
axes[1].set_ylabel('Kumulativ varians (%)')
axes[1].legend()

plt.tight_layout()
plt.savefig('outputs/figures/03_pca_scree.png', dpi=150)
plt.show()

print(f'\nAntal komponenter för 90% förklarad varians: {n_komp_90}')

In [ ]:
# Skapa PCA-transformerade dataset
pca = PCA(n_components=n_komp_90, random_state=42)
X_train_pca = pca.fit_transform(X_train)  # Fit + transform på träningsdata
X_test_pca  = pca.transform(X_test)       # Bara transform på testdata

print(f'PCA: {X_train.shape[1]} → {X_train_pca.shape[1]} dimensioner')

## PCA + Logistisk Regression

In [ ]:
utvardera(
    f'LR + PCA ({n_komp_90} komp.)',
    LogisticRegression(max_iter=1000, random_state=42),
    X_train_pca, X_test_pca, y_train, y_test
)

## PCA + KNN

In [ ]:
utvardera(
    f'KNN (k={basta_k}) + PCA ({n_komp_90} komp.)',
    KNeighborsClassifier(n_neighbors=basta_k),
    X_train_pca, X_test_pca, y_train, y_test
)

---

## UMAP – Uniform Manifold Approximation and Projection

UMAP är en icke-linjär dimensionsreduktionsmetod som försöker bevara den lokala
topologiska strukturen i data. Den är bra på att visualisera komplexa mönster
som PCA (som är linjär) kan missa.

Vi använder UMAP på två sätt:
1. **Visualisering** – 2D-scatter plot för att se om klasserna är separerbara
2. **Som features** – träna klassificerare på de reducerade dimensionerna

In [ ]:
# UMAP för visualisering (2D)
umap_2d = UMAP(n_components=2, random_state=42, n_neighbors=30, min_dist=0.1)
X_train_umap2d = umap_2d.fit_transform(X_train)

fig, ax = plt.subplots(figsize=(9, 7))
farger_scatter = ['steelblue' if y == 0 else 'tomato' for y in y_train]
ax.scatter(X_train_umap2d[:, 0], X_train_umap2d[:, 1],
           c=farger_scatter, alpha=0.6, edgecolors='white', linewidths=0.3, s=50)
ax.set_title('UMAP 2D – Träningsdata\n(blå = Frisk, röd = Sjuk)', fontweight='bold')
ax.set_xlabel('UMAP dim 1')
ax.set_ylabel('UMAP dim 2')
plt.tight_layout()
plt.savefig('outputs/figures/03_umap_2d.png', dpi=150)
plt.show()

In [ ]:
# UMAP för klassificering – vi testar med fler dimensioner
umap_klass = UMAP(n_components=10, random_state=42, n_neighbors=30, min_dist=0.1)
X_train_umap = umap_klass.fit_transform(X_train)
X_test_umap  = umap_klass.transform(X_test)
print(f'UMAP: {X_train.shape[1]} → {X_train_umap.shape[1]} dimensioner')

## UMAP + Logistisk Regression

In [ ]:
utvardera(
    'LR + UMAP (10 komp.)',
    LogisticRegression(max_iter=1000, random_state=42),
    X_train_umap, X_test_umap, y_train, y_test
)

## UMAP + KNN

In [ ]:
utvardera(
    f'KNN (k={basta_k}) + UMAP (10 komp.)',
    KNeighborsClassifier(n_neighbors=basta_k),
    X_train_umap, X_test_umap, y_train, y_test
)

## Spara modeller och resultat

In [ ]:
# Spara tränade modeller för att kunna ladda dem i Notebook 4
for namn, data in resultat.items():
    filnamn = namn.replace(' ', '_').replace('/', '-').replace('(', '').replace(')', '')
    filnamn = filnamn.replace('=', '').replace('.', '') + '.pkl'
    joblib.dump(data['modell'], f'outputs/models/{filnamn}')

# Spara resultat-tabell som CSV
df_alla_resultat = pd.DataFrame([
    {k: v for k, v in v_dict.items() if k not in ['modell', 'y_pred', 'y_prob']}
    for v_dict in resultat.values()
], index=resultat.keys())
df_alla_resultat.to_csv('outputs/models/results.csv')

print('Sparade modeller och resultat:')
print(df_alla_resultat[['Accuracy', 'F1-score', 'ROC-AUC', 'CV F1 (5-fold)']].round(4).to_string())

## Sammanfattning Modellering

### Vad testade vi och vad lärde vi oss?

Vi tränade **8 modeller** i totalt:
- 4 grundmodeller (LR, KNN, Beslutsträd, Random Forest)
- 2 modeller med PCA (LR+PCA, KNN+PCA)
- 2 modeller med UMAP (LR+UMAP, KNN+UMAP)

Frågor att fundera på inför Notebook 4:
- Vilken modell presterar bäst? Är det konsekvent över alla mätvärden?
- Hjälper PCA eller UMAP prestandan, eller försämrar de den?
- Finns det en trade-off mellan tolkbarhet och prestanda?

---
**Nästa steg → Notebook 4: Utvärdering & Slutsatser**
Vi jämför alla modeller visuellt, ritar ROC-kurvor och besvarar
projektets kärnfråga: *Kan modellen förbättras?*